# 02 - Análisis Descriptivo y Pruebas de Normalidad

El objetivo de este notebook es describir la muestra del estudio de osteoporosis y probar los supuestos de distribución de las variables continuas.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import shapiro

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
PROCESSED_DATA_DIR = Path(
    "/home/marcos-maravilla/análisis_estadístico_osteoporosis/data/processed"
)
PARQUET_PATH = PROCESSED_DATA_DIR / "BD_Clean_Osteoporosis.parquet"
PICKLE_PATH = PROCESSED_DATA_DIR / "BD_Clean_Osteoporosis.pkl"

if PARQUET_PATH.exists():
    try:
        df = pd.read_parquet(PARQUET_PATH)
        loaded_path = PARQUET_PATH
    except Exception as error:
        print("No se pudo cargar el archivo Parquet; se intentará cargar el archivo Pickle.")
        print(f"Detalle del error: {type(error).__name__}: {error}")
        if not PICKLE_PATH.exists():
            raise FileNotFoundError(f"No se encontró archivo alternativo: {PICKLE_PATH}") from error
        df = pd.read_pickle(PICKLE_PATH)
        loaded_path = PICKLE_PATH
elif PICKLE_PATH.exists():
    df = pd.read_pickle(PICKLE_PATH)
    loaded_path = PICKLE_PATH
else:
    raise FileNotFoundError(
        f"No se encontró ningún archivo procesado en {PROCESSED_DATA_DIR}"
    )

print(f"Dataset cargado desde: {loaded_path}")
print(f"Dimensiones del dataset: {df.shape[0]:,} filas y {df.shape[1]:,} columnas")

## 1. Pruebas de Normalidad (Shapiro-Wilk)

In [ ]:
continuous_variables = ["edad", "peso_(kg)", "altura_(cm)", "imc"]

for variable in continuous_variables:
    values = pd.to_numeric(df[variable], errors="coerce").dropna()
    statistic, p_value = shapiro(values)
    interpretation = "Distribución No Normal" if p_value < 0.05 else "Distribución Normal"

    print(f"{variable}")
    print(f"  W = {statistic:.4f}")
    print(f"  p-valor = {p_value:.4g}")
    print(f"  Interpretación: {interpretation}\n")

## 2. Estadística Descriptiva

In [ ]:
descriptive_continuous = []

for variable in continuous_variables:
    values = pd.to_numeric(df[variable], errors="coerce").dropna()
    p25 = values.quantile(0.25)
    median = values.median()
    p75 = values.quantile(0.75)

    descriptive_continuous.append(
        {
            "variable": variable,
            "mediana": median,
            "p25": p25,
            "p75": p75,
            "rango_intercuartilico": f"{p25:.2f} - {p75:.2f}",
        }
    )

continuous_summary = pd.DataFrame(descriptive_continuous)
continuous_summary

In [ ]:
categorical_variables = ["sexo", "alteracion_osea", "trabaja", "realiza_af"]

for variable in categorical_variables:
    frequency_table = (
        df[variable]
        .value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="frecuencia_absoluta")
    )
    frequency_table["porcentaje"] = (
        frequency_table["frecuencia_absoluta"] / len(df) * 100
    ).round(2)

    print(f"\nVariable: {variable}")
    display(frequency_table)